In [1]:
import tensorflow as tf

# 시스템에 장착된 GPU 장치 목록 가져오기
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    try:
        for gpu in gpus:
            # GPU 메모리 동적 할당 설정 (이미지 속 오타 'experimntal' 수정 포함)
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

I0000 00:00:1791684932.644777 1684146 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1791684932.691217 1684146 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1791684933.619245 1684146 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1791684934.812106 1684146 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries co

In [2]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, AveragePooling2D
from tensorflow.keras.layers import Flatten, Dense, Input

class LeNet:
    @staticmethod
    def build(input_shape=(32,32,1), activation='sigmoid'):
        model = Sequential()
        model.add(Input(shape=input_shape))

        # 첫 번째 CONV => ACTIVATION => POOL 계층
        model.add(Conv2D(6, 5, activation=activation,
                        kernel_initializer='random_uniform'))
        model.add(AveragePooling2D(pool_size=(2,2)))

        # 두 번째 CONV => ACTIVATION => POOL 계층
        model.add(Conv2D(16, 5, activation=activation,
                        kernel_initializer='random_uniform'))
        model.add(AveragePooling2D(pool_size=(2,2)))

        # 첫 번째 FC 계층
        model.add(Flatten())
        model.add(Dense(120, activation=activation))

        # 두 번째 FC 계층
        model.add(Dense(84, activation=activation))

        # 출력층 soft-max 활성화 함수 사용
        model.add(Dense(10, activation='softmax'))
        return model


In [3]:
model = LeNet.build(input_shape=(28, 28, 1), activation="relu")
model.summary()

W0000 00:00:1791684934.859287 1684146 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1791684934.974044 1684146 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 951 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5090, pci bus id: 0000:c2:00.0, compute capability: 12.0a


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 24, 24, 6)      │           156 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 12, 12, 6)      │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 8, 8, 16)       │         2,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d_1             │ (None, 4, 4, 16)       │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 84)             │        10,164 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           850 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 44,426 (173.54 KB)

 Trainable params: 44,426 (173.54 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd", metrics=['accuracy'])

In [5]:
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = mnist.load_data()
X_train = X_train.reshape(-1, 28, 28, 1) / 255.0
X_test = X_test.reshape(-1, 28, 28, 1) / 255.0

In [6]:
model.fit(X_train, y_train, batch_size=200, epochs=20)

Epoch 1/20


I0000 00:00:1791685078.743760 1684675 service.cc:153] XLA service 0x7047c802fe10 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791685078.743801 1684675 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5090, Compute Capability 12.0a (Driver: 13.0.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.19.0)
I0000 00:00:1791685078.772462 1684675 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791685078.845297 1684675 cuda_dnn.cc:461] Loaded cuDNN version 91900
I0000 00:00:1791685078.865698 1684675 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_701__.23
I0000 00:00:1791685079.672789 1684675 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0

 75/300 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.1689 - loss: 2.3009

I0000 00:00:1791685083.322256 1684675 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


300/300 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.1864 - loss: 2.2942
Epoch 2/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3958 - loss: 2.0767
Epoch 3/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8271 - loss: 0.6137
Epoch 4/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8817 - loss: 0.3950
Epoch 5/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9008 - loss: 0.3299
Epoch 6/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9144 - loss: 0.2881
Epoch 7/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9214 - loss: 0.2595
Epoch 8/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9278 - loss: 0.2371
Epoch 9/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9336 - loss: 0.2192
Epoch 10/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9383 - loss: 0.2042
Epoch 11/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9424 - loss: 0.1919
Epoch 12/20
300/300 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy: {accuracy * 100:.2f}%")

I0000 00:00:1791685133.111721 1684674 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1791685133.284368 1693243 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 4 bytes spill stores, 4 bytes spill loads

I0000 00:00:1791685133.484654 1693210 subprocess_compilation.cc:348] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_MatMul_8', 56 bytes spill stores, 56 bytes spill loads

I0000 00:00:1791685134.705549 1684674 dot_search_space.cc:240] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
I0000 00:00:1791685135.017260 

Accuracy: 96.23%


: 